In [0]:
CREATE SCHEMA IF NOT EXISTS nyc_taxi.silver;

In [0]:
%python
df_bronze=spark.table("nyc_taxi.bronze.yellow_trip_raw")

In [0]:
%python
silver_df=df_bronze.filter(
    (df_bronze.trip_distance>0) &
    (df_bronze.fare_amount>0) &
    (df_bronze.total_amount>=0)
)
display(silver_df)

In [0]:
%python
from pyspark.sql.functions import coalesce, lit

silver_df = (
    silver_df
    .withColumn("passenger_count", coalesce("passenger_count", lit(0)))
    .withColumn("RatecodeID", coalesce("RatecodeID", lit(0)))
    .withColumn("congestion_surcharge", coalesce("congestion_surcharge", lit(0)))
    .withColumn("Airport_fee", coalesce("Airport_fee", lit(0)))
    .fillna({"store_and_fwd_flag": "Unknown"})
)
display(silver_df.count())

In [0]:
%python

from pyspark.sql.functions import coalesce, lit

silver_df = (
    silver_df
    .withColumn("passenger_count", coalesce("passenger_count", lit(0)))
    .withColumn("RatecodeID", coalesce("RatecodeID", lit(0)))
    .withColumn("congestion_surcharge", coalesce("congestion_surcharge", lit(0)))
    .withColumn("Airport_fee", coalesce("Airport_fee", lit(0)))
    .fillna({"store_and_fwd_flag": "Unknown"})
)

In [0]:
%python
display(silver_df.count())
#After handing missing value


In [0]:
%python
from pyspark.sql.functions import col, count, when
missing =silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
])
display(missing)

In [0]:
%python
print("Total:", silver_df.count())
print("Distinct:", silver_df.distinct().count())

In [0]:
%python
silver_df.printSchema()

In [0]:
%python
from pyspark.sql.functions import expr, hour, to_date

silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    expr("timestampdiff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime)")
)

silver_df = silver_df.withColumn(
    "trip_date",
    to_date("tpep_pickup_datetime")
)

silver_df = silver_df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

In [0]:
%python
display(
    silver_df.select(
        "trip_duration_minutes",
        "trip_date",
        "pickup_hour"
    ).limit(10)
)

In [0]:
%python
silver_df.filter("trip_duration_minutes <= 0").count()

In [0]:
%python
silver_df = silver_df.filter(
    silver_df.trip_duration_minutes > 0
)

In [0]:
%python
print("Final Silver Rows:", silver_df.count())

In [0]:
%python
total_rows = silver_df.count()
distinct_rows = silver_df.distinct().count()

print(total_rows)
print(distinct_rows)

In [0]:
%python
silver_df.filter("trip_duration_minutes > 1440").count()

In [0]:
%python
silver_df.filter("trip_distance > 200").count()

In [0]:
%python
from pyspark.sql.functions import month

silver_df = silver_df.withColumn(
    "trip_month",
    month("trip_date")
)

In [0]:
%python
from pyspark.sql.functions import date_format

silver_df = silver_df.withColumn(
    "day_of_week",
    date_format("trip_date", "EEEE")
)

In [0]:
%python
from pyspark.sql.functions import year

silver_df = silver_df.filter(
    (year("tpep_pickup_datetime") >= 2020) &
    (year("tpep_pickup_datetime") <= 2026)
)

In [0]:
%python


In [0]:
%python

silver_df.select(
    "tpep_pickup_datetime"
).orderBy("tpep_pickup_datetime").show(10)

In [0]:
%python
from pyspark.sql.functions import current_date, to_date

silver_df = silver_df.filter(
    to_date("tpep_pickup_datetime") <= current_date()
)

In [0]:
%python
silver_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("nyc_taxi.silver.yellow_trip_clean")